In [ ]:
extern crate onq;
use onq::{
    Circuit, CircuitBuilder, Operation, QduId, Simulator, SimulationResult, StableState, OnqError
};
use std::collections::HashMap;
println!("onq types imported successfully for Grover's Algorithm.");

In [ ]:
// Helper for QduId creation
fn qid(id: u64) -> QduId { QduId(id) }

// Define the two QDUs involved in our 2-qubit search
let q0 = qid(0); // QDU 0
let q1 = qid(1); // QDU 1

println!("QDUs defined: q0={}, q1={}", q0, q1);

In [ ]:
// --- Build Circuit using Grover's Search Logic ---

let mut builder = CircuitBuilder::new();
println!("Building 2-QDU Grover's Algorithm Circuit...");

// 1. Initialization: Put both QDUs in equal superposition |+>
builder = builder.add_op(Operation::InteractionPattern {
    target: q0,
    pattern_id: "Superposition".to_string(), // H analog
});
builder = builder.add_op(Operation::InteractionPattern {
    target: q1,
    pattern_id: "Superposition".to_string(),
});
println!("  Step 1: Initialized QDUs in equal superposition.");

// 2. Oracle: Mark the |11> state
// A Controlled-Z (PhaseIntroduce) flips the phase only when both are |1>
builder = builder.add_op(Operation::ControlledInteraction {
    control: q0,
    target: q1,
    pattern_id: "PhaseIntroduce".to_string(), // CZ analog
});
println!("  Step 2: Applied Oracle to mark |11> state.");

// 3. Diffuser (Amplitude Amplification): Inversion about the mean
// Apply Superposition (H)
builder = builder.add_op(Operation::InteractionPattern {
    target: q0,
    pattern_id: "Superposition".to_string(),
});
builder = builder.add_op(Operation::InteractionPattern {
    target: q1,
    pattern_id: "Superposition".to_string(),
});
// Apply QualityFlip (X)
builder = builder.add_op(Operation::InteractionPattern {
    target: q0,
    pattern_id: "QualityFlip".to_string(),
});
builder = builder.add_op(Operation::InteractionPattern {
    target: q1,
    pattern_id: "QualityFlip".to_string(),
});
// Apply PhaseIntroduce (CZ analog)
builder = builder.add_op(Operation::ControlledInteraction {
    control: q0,
    target: q1,
    pattern_id: "PhaseIntroduce".to_string(),
});
// Apply QualityFlip (X)
builder = builder.add_op(Operation::InteractionPattern {
    target: q0,
    pattern_id: "QualityFlip".to_string(),
});
builder = builder.add_op(Operation::InteractionPattern {
    target: q1,
    pattern_id: "QualityFlip".to_string(),
});
// Apply Superposition (H)
builder = builder.add_op(Operation::InteractionPattern {
    target: q0,
    pattern_id: "Superposition".to_string(),
});
builder = builder.add_op(Operation::InteractionPattern {
    target: q1,
    pattern_id: "Superposition".to_string(),
});
println!("  Step 3: Applied Diffuser sequence to amplify marked state.");

// 4. Stabilize both QDUs to observe the final result
builder = builder.add_op(Operation::Stabilize {
    targets: vec![q0, q1],
});
println!("  Step 4: Added final stabilization.");

let circuit = builder.build();

// Print the constructed circuit diagram
println!("\nGrover's Algorithm Circuit (2-QDU):\n{}", circuit);

In [ ]:
// --- Run Simulation ---
let simulator = Simulator::new();
println!("\nRunning simulation...");

match simulator.run(&circuit) {
    Ok(result) => {
        println!("Simulation finished successfully.");
        println!("\nSimulation Result Details:");
        println!("{}", result);

        // --- Basic Result Analysis ---
        // For a 2-QDU Grover search marking |11>, the ideal theoretical probability
        // of measuring 11 after one iteration is 100%.
        // Let's verify how the onq framework stabilizes this state.
        
        println!("\nAnalysis:");
        let mut outcomes = HashMap::new();
        
        if let Some(StableState::ResolvedQuality(q0_outcome)) = result.get_stable_state(&q0) {
             outcomes.insert("q0", q0_outcome);
             println!("- QDU 0 ({}) stabilized to state: {}", q0, q0_outcome);
        } else {
             println!("- QDU 0 ({}) was not found in stabilization results.", q0);
        }
        
        if let Some(StableState::ResolvedQuality(q1_outcome)) = result.get_stable_state(&q1) {
             outcomes.insert("q1", q1_outcome);
             println!("- QDU 1 ({}) stabilized to state: {}", q1, q1_outcome);
        } else {
             println!("- QDU 1 ({}) was not found in stabilization results.", q1);
        }
        
        if outcomes.get("q0") == Some(&&1_u64) && outcomes.get("q1") == Some(&&1_u64) { // Assuming 1.0 represents |1>
            println!("\nSuccess: The marked state |11> was successfully amplified and resolved!");
        } else {
            println!("\nNote: The state stabilized to something other than pure |11>. Check your deterministic stabilization constants.");
        }
    }
    Err(e) => {
        eprintln!("\n--- Simulation Failed ---");
        eprintln!("Error: {}", e);
    }
}